In [34]:
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import corner
import asdf
from tqdm import tqdm
import subprocess

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


GALAXY_ID = TABLE["SURVEY_ID"]        # object ID, used for labeling/output files
SERSIC_FILTERS = ["F444W","F356W","F277W"]                    # filter name, used for labeling/output files
SURVEY = TABLE["SURVEY"]

CUTOUT_SIZE = 0.96 #as
PIXEL_SIZE = 0.03 #as

In [35]:
def run_fits():
    """
    Runs morphometrica fits on three filters for each of the galaxies.

    Parameters
    ----------
    wanted : list of str
        List of parameters I want to extract from the fit results.

    Returns
    -------
    table : table of floats
        Table of data with my wanted parameter results.
    """
    rows = []


    for i in tqdm(range(len(GALAXY_ID[0:2])),total=len(GALAXY_ID[0:2])):
            for filt in SERSIC_FILTERS:

                try:
                    science_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts/{GALAXY_ID[i]}/{filt}.fits"

                    psf_path = f"/nvme/scratch/work/alberttg/Summer_project/PSFs/{SURVEY[i]}/{filt}_psf_norm.fits"
                    psf_ext = 0

                    output_dir = f"/nvme/scratch/work/alberttg/Summer_project/Morphometrica_data/{GALAXY_ID[i]}"

                    result = subprocess.run(f"python /nvme/scratch/work/westcottl/Codes/Morfometryka/Code/morfometryka965.py {science_path} {psf_path} noshow rerun", 
                                   shell=True, capture_output=True, text=True)
                    
                    lines = result.stdout.splitlines()

                    header = None
                    values = None

                    for j, line in enumerate(lines):
                        if line.startswith("# rootname9.65"):
                            header = [x.strip() for x in line[1:].split(",")]
                            values = [x.strip() for x in lines[j+1].split(",")]
                            break

                    if header is not None:
                        row = dict(zip(header, values))
                        row["Galaxy_ID"] = GALAXY_ID[i]
                        row["Filter"] = filt
                        rows.append(row)
                
                except Exception as e:
                    print(e)
                    continue

    table = pd.DataFrame(rows)
    return table

In [36]:
if __name__ == "__main__":
    table = run_fits()
    print(table.colnames)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:30<00:00, 15.37s/it]


AttributeError: 'DataFrame' object has no attribute 'colnames'